# It's Alive - Full Bielik 1.5B Model on Triton Kernels

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/qooba/bielik-anatomy-triton/blob/main/notebooks/ep07-its-alive.ipynb)

Part of the [Bielik Anatomy](https://github.com/qooba/bielik-anatomy-triton) series - Episode 7: It's Alive. Full write-up: [ep07-its-alive.md](https://github.com/qooba/bielik-anatomy-triton/blob/main/docs/ep07-its-alive.md).

Every kernel built across episodes 1-6 (matmul, RMSNorm, RoPE, Flash Attention, SwiGLU) plus two new ones (embedding lookup, bias support in matmul/SwiGLU) is wired together into a complete, runnable [Bielik-1.5B-v3.0-Instruct](https://huggingface.co/speakleash/Bielik-1.5B-v3.0-Instruct) model. Pretrained HuggingFace weights are loaded via SafeTensors and the model generates real Polish text - ~100% Triton compute, no PyTorch `nn.Module` layers.

**Runtime:** select a GPU runtime before running: `Runtime -> Change runtime type -> T4 GPU` (free tier works). You'll also download ~3 GB of model weights from Hugging Face, so this notebook takes a few minutes to get to its first generated token.

In [ ]:
!nvidia-smi

In [ ]:
import os

REPO_DIR = "bielik-anatomy-triton"
if not os.path.isdir(REPO_DIR):
    !git clone --quiet https://github.com/qooba/bielik-anatomy-triton {REPO_DIR}
%cd {REPO_DIR}
!pip install -q -r requirements.txt
# Not pinned in requirements.txt (only used by the bielik/ package, not the kernel benchmarks):
!pip install -q transformers safetensors huggingface_hub

## Note on dtype (T4 vs. the docs)

The episode doc benchmarks this model in **bfloat16** on an RTX 4060 Ti (Ada), and this notebook loads it in bfloat16 too, even on T4. That's deliberate, not just matching the docs: Bielik (like most LLaMA/Mistral-family models) was trained in bf16, and transformer residual streams routinely develop large-magnitude activations in deeper layers. Every kernel here (RMSNorm, matmul+bias, attention, SwiGLU) accumulates in fp32 internally but stores its output back down to the model dtype after each op, 32 layers deep. bf16's exponent range (same as fp32, ~3.4e38) comfortably absorbs that; fp16's much smaller range (max ~65504) does not — an early version of this notebook used float16 for T4's native FP16 tensor cores, and it produced fluent-looking but **completely incoherent** output (real Polish words, no meaning), which is what fp16 overflow cascading through 32 layers looks like in practice, not a sampling/temperature problem.

T4 (Turing, compute capability 7.5) has no native BF16 tensor cores, so generation here runs through a slower, non-tensor-core path and will be noticeably slower than the doc's numbers (T4 vs. a dedicated Ada GPU, plus Colab's shared/throttled GPUs, plus no KV cache yet — see the doc). Keep `max_new_tokens` modest (the defaults below) unless you like waiting. If you have an Ampere+ runtime (A100/L4 via Colab Pro), bf16 there gets real tensor-core acceleration with no correctness trade-off.

## 1. Download model weights

Downloads the Hugging Face checkpoint into the standard HF cache (`~/.cache/huggingface/hub`), where `BielikModel.from_pretrained` and `AutoTokenizer.from_pretrained` will find it below. This is a public model, so no login/token should be required; if you hit a 401/403, accept the model's terms on its Hugging Face page and run `huggingface_hub.login()` first.

In [ ]:
from huggingface_hub import snapshot_download

MODEL_ID = "speakleash/Bielik-1.5B-v3.0-Instruct"

snapshot_download(repo_id=MODEL_ID)
print("Weights downloaded.")

## 2. Load the model and tokenizer

In [ ]:
import torch
from transformers import AutoTokenizer
from bielik import BielikModel

DEVICE = "cuda"
DTYPE = torch.bfloat16  # required for correctness on this model, even though T4 (Turing) has to run it without native BF16 tensor cores - see the note above

model = BielikModel.from_pretrained(MODEL_ID, device=DEVICE, dtype=DTYPE)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
print(model)

## 3. Generate a response

Single-shot generation (greedy/sampled, no KV cache) through the Triton-kernel model, timed so you can see tokens/sec on whatever GPU Colab gave you.

In [ ]:
import time

messages = [{"role": "user", "content": "Napisz krotki wiersz o polskiej jesieni."}]
if tokenizer.chat_template:
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
else:
    prompt = messages[0]["content"]

input_ids = tokenizer.encode(prompt, return_tensors="pt").to(DEVICE)
print(f"Prompt tokens: {input_ids.shape[1]}")

start = time.time()
output_ids = model.generate(
    input_ids,
    max_new_tokens=64,
    temperature=0.8,
    top_k=50,
    top_p=0.95,
    do_sample=True,
)
elapsed = time.time() - start

new_tokens = output_ids.shape[1] - input_ids.shape[1]
response = tokenizer.decode(output_ids[0, input_ids.shape[1]:], skip_special_tokens=True)

print(response)
print(f"\n{new_tokens} tokens in {elapsed:.1f}s -> {new_tokens / elapsed:.2f} tok/s")

## 4. Interactive chat (optional)

Reuses the already-loaded `model`/`tokenizer` for a multi-turn conversation, keeping history and applying the tokenizer's chat template each turn. Run the cell, type a message, and press Enter; type `exit` (or leave it blank) to stop.

In [ ]:
conversation = []

def chat_turn(user_message, max_new_tokens=100, temperature=0.8, top_k=50, top_p=0.95):
    conversation.append({"role": "user", "content": user_message})
    if tokenizer.chat_template:
        prompt = tokenizer.apply_chat_template(
            conversation, tokenize=False, add_generation_prompt=True
        )
    else:
        prompt = user_message

    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(DEVICE)
    output_ids = model.generate(
        input_ids,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        top_k=top_k,
        top_p=top_p,
        do_sample=True,
    )
    response = tokenizer.decode(
        output_ids[0, input_ids.shape[1]:], skip_special_tokens=True
    ).strip()
    conversation.append({"role": "assistant", "content": response})
    return response

while True:
    user_message = input("You: ").strip()
    if user_message.lower() in ("exit", "quit", "q", ""):
        break
    print(f"Bielik: {chat_turn(user_message)}\n")

---
[Full episode write-up](https://github.com/qooba/bielik-anatomy-triton/blob/main/docs/ep07-its-alive.md) · [Series overview](https://github.com/qooba/bielik-anatomy-triton/blob/main/README.md)